# BTC 15m Strategy Backtest

This notebook backtests the current model + Kelly sizing stack on historical Coinbase 15-minute BTC candles.

Notes:
- Order-book/funding/liquidation/news features are set to neutral in this historical test unless you merge external histories.
- Market odds default to fixed values unless you provide an odds CSV.


In [ ]:
import pandas as pd
import plotly.express as px

from btc15m import CoinbaseClient, BacktestConfig, run_backtest, summarize_backtest


In [ ]:
# Configuration
PRODUCT_ID = "BTC-USD"
CANDLE_LIMIT = 1200  # ~12.5 days of 15m candles

config = BacktestConfig(
    initial_bankroll=1000.0,
    market_up_price=0.50,
    market_down_price=0.50,
    fee_rate=0.0156,
    edge_buffer=0.02,
    fractional_kelly=0.50,
    max_fraction=0.20,
    min_history=40,
)

ODDS_CSV_PATH = None  # e.g. "historical_odds.csv" with timestamp,up_price,down_price


In [ ]:
client = CoinbaseClient(timeout=10)
candles_15m = client.fetch_candles(product_id=PRODUCT_ID, granularity=900, limit=CANDLE_LIMIT)
candles_15m.tail()


In [ ]:
odds_df = None
if ODDS_CSV_PATH:
    odds_df = pd.read_csv(ODDS_CSV_PATH)

results = run_backtest(candles_15m=candles_15m, config=config, odds_df=odds_df)
stats = summarize_backtest(results, initial_bankroll=config.initial_bankroll)
stats


In [ ]:
results.tail()


In [ ]:
fig_equity = px.line(results, x="window_end", y="bankroll", title="Equity Curve")
fig_equity.show()

traded = results[results['decision'].isin(['UP', 'DOWN'])].copy()
if not traded.empty:
    fig_ret = px.histogram(traded, x="trade_return_pct", nbins=50, title="Trade Return Distribution")
    fig_ret.show()


In [ ]:
# Quick diagnostics
traded = results[results['decision'].isin(['UP', 'DOWN'])].copy()
print('Total rows:', len(results))
print('Traded rows:', len(traded))
print('No-trade rows:', int((results['decision'] == 'NO_TRADE').sum()))

if not traded.empty:
    print('Average stake fraction:', traded['stake_fraction'].mean())
    print('Worst 5 trades by pnl:')
    display(traded.sort_values('pnl_usd').head(5))
